In [17]:
%%writefile requirements.txt
pandas==2.2.2
requests==2.31.0
fastapi==0.110.0
uvicorn==0.28.0
streamlit==1.32.0
pytest==8.1.1

Overwriting requirements.txt


### Configuración: `requirements.txt`

Este bloque define las dependencias del proyecto en `requirements.txt`, asegurando que todas las librerías necesarias estén disponibles para la ejecución del código, tanto localmente como en entornos dockerizados.

In [2]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["sh", "-c", "python etl/pipeline.py && uvicorn api.main:app --host 0.0.0.0 --port 8000"]

Writing Dockerfile


### Configuración: `Dockerfile`

Aquí se define el `Dockerfile` para construir una imagen Docker de la aplicación. Incluye la instalación de dependencias, la copia de archivos y la configuración del comando de inicio para ejecutar el ETL y la API FastAPI.

In [3]:
import os


os.makedirs("data", exist_ok=True)
os.makedirs("etl", exist_ok=True)
os.makedirs('dashboards', exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs("docker", exist_ok=True)
os.makedirs("api", exist_ok=True)

### Inicialización de Directorios

Este código crea las estructuras de directorios necesarias (`data`, `etl`, `api`, `tests`) para organizar el proyecto y almacenar los archivos generados por el pipeline y la API.

In [4]:
%%writefile etl/pipeline.py
import os
import logging
import requests
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

Writing etl/pipeline.py


### Creación Inicial de `etl/pipeline.py`

Este es el primer paso para crear el archivo `etl/pipeline.py`, inicializándolo con importaciones básicas y configuración de logging. Las versiones posteriores del archivo lo sobreescribirán con la lógica completa.

In [5]:
import requests

url_login = "https://api.cne.cl/api/login"
payload = {
    "email": "je.urquiaga@duocuc.cl",
    "password": "Joder123#"
}
response = requests.post(url_login, data=payload)

### Prueba de Conexión a la API de Login de la CNE

Este bloque de código realiza una prueba simple de conexión al endpoint de login de la API de la CNE para verificar que se puede establecer comunicación y se reciben credenciales.

## etl/pipeline.py - Pipeline de Extracción, Transformación y Carga (ETL)

In [11]:
%%writefile etl/pipeline.py
import os
import logging
import requests
import pandas as pd
import urllib3

# Desactivamos advertencias de certificados SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

class DataPipeline:
    def __init__(self):
        # 1. Variables de entorno para ocultar las credenciales en Docker
        self.email = os.getenv("CNE_USER", "je.urquiaga@duocuc.cl")
        self.password = os.getenv("CNE_PASSWORD", "Joder123#")

        self.url_login = "https://api.cne.cl/api/login"
        # 2. URL actualizada a la versión 4
        self.cne_url = "https://api.cne.cl/api/v4/estaciones"

        # Asegúrate de que este nombre coincida exactamente con tu archivo físico
        self.csv_path = "df_bencinera_final.csv"
        self.token = None

    def obtener_token(self):
        logging.info("Iniciando Handshake con API CNE (POST)...")
        payload = {"email": self.email, "password": self.password}
        try:
            res = requests.post(self.url_login, data=payload, timeout=10, verify=False)
            res.raise_for_status()
            self.token = res.json().get("token")
            if self.token:
                logging.info("Autenticación exitosa. Token capturado en memoria.")
            else:
                logging.error("Login devolvió 200 OK, pero no se halló el token.")
        except Exception as e:
            logging.error(f"Falla en autenticación: {e}")

    def procesar_csv_local(self):
        logging.info("Procesando histórico de transacciones (CSV)...")
        try:
            df = pd.read_csv(self.csv_path)
            cols = ['id_vehiculo', 'industria', 'nombre_prod', 'cantidad', 'fecha']
            # Validación de columnas antes de filtrar
            columnas_presentes = [c for c in cols if c in df.columns]
            df = df[columnas_presentes].dropna(subset=['cantidad'])
            df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')

            os.makedirs("data", exist_ok=True)
            df.to_csv("data/transacciones_limpias.csv", index=False)
            logging.info("Dataset CSV limpio exportado a /data.")
        except Exception as e:
            logging.error(f"Falla al procesar CSV: {e}")

    def procesar_api_cne(self):
        logging.info("Extrayendo telemetría de precios CNE v4 (GET)...")
        df = pd.DataFrame()

        try:
            if not self.token:
                raise ValueError("Sin token. Abortando extracción en vivo.")

            parametros_url = {"token": self.token}
            res = requests.get(self.cne_url, params=parametros_url, timeout=30, verify=False)
            res.raise_for_status()

            if res.text.strip().startswith("<!DOCTYPE html>"):
                raise ValueError("El servidor retornó HTML. Revisa la documentación de la API.")

            data_json = res.json()
            data = data_json.get("data", data_json) if isinstance(data_json, dict) else data_json

            # 3. Aplicación de json_normalize para aplanar las jerarquías
            if data:
                df = pd.json_normalize(data)
                logging.info("API consumida y JSON aplanado exitosamente.")
            else:
                logging.warning("La API de la CNE respondió, pero la lista de datos está vacía.")

        except Exception as e:
            logging.error(f"Error de red/API externa: {e}. Activando Fallback al CSV local...")
            try:
                # 4. Fallback al archivo CSV actualizado
                df = pd.read_csv("df_datos_estaciones_cne_2.csv")
                logging.info("Respaldo local cargado correctamente.")
            except FileNotFoundError:
                logging.critical("CRÍTICO: No se pudo conectar a la API ni cargar el archivo de respaldo.")
                return

        if not df.empty:
            # 5. Mapeo adaptado a la estructura anidada y separada por puntos
            cols_to_rename = {
                'ubicacion.nombre_comuna': 'comuna',
                'precios.93.precio': 'precio_93',
                'precios.95.precio': 'precio_95',
                'precios.petroleo_diesel.precio': 'diesel'
            }

            # Renombramos solo las columnas que existan en el DataFrame aplanado
            df_fil = df.rename(columns={k: v for k, v in cols_to_rename.items() if k in df.columns})

            # Seleccionamos exclusivamente nuestras columnas objetivo
            columnas_finales = [c for c in ['comuna', 'precio_93', 'precio_95', 'diesel'] if c in df_fil.columns]
            df_fil = df_fil[columnas_finales]

            for col in ['precio_93', 'precio_95', 'diesel']:
                if col in df_fil.columns:
                    df_fil[col] = pd.to_numeric(df_fil[col], errors='coerce')

            os.makedirs("data", exist_ok=True)
            df_fil.to_csv("data/precios_cne.csv", index=False)
            logging.info("Precios CNE estructurados y sincronizados en /data.")

    def ejecutar(self):
        self.obtener_token()
        self.procesar_csv_local()
        self.procesar_api_cne()

if __name__ == "__main__":
    DataPipeline().ejecutar()

Overwriting etl/pipeline.py


### Definición Completa del ETL (`etl/pipeline.py`)

Este es el código completo del pipeline ETL, el cual se encarga de: 1. Autenticarse con la API de la CNE para obtener un token. 2. Procesar un archivo CSV local de transacciones. 3. Extraer y limpiar datos de precios de combustibles desde la API de la CNE. Todo el proceso incluye manejo de errores y logging detallado.

In [12]:
!python etl/pipeline.py

2026-06-23 01:24:52,435 [INFO] Iniciando Handshake con API CNE (POST)...
2026-06-23 01:24:53,251 [INFO] Autenticación exitosa. Token capturado en memoria.
2026-06-23 01:24:53,251 [INFO] Procesando histórico de transacciones (CSV)...
2026-06-23 01:24:54,949 [INFO] Dataset CSV limpio exportado a /data.
2026-06-23 01:24:54,954 [INFO] Extrayendo telemetría de precios CNE v4 (GET)...
2026-06-23 01:24:56,493 [INFO] API consumida y JSON aplanado exitosamente.
2026-06-23 01:24:56,506 [INFO] Precios CNE estructurados y sincronizados en /data.


### Ejecución del Pipeline ETL

Este comando ejecuta el script `etl/pipeline.py`, iniciando el proceso de extracción, transformación y carga de datos que genera los archivos CSV limpios en el directorio `data`.

In [13]:
%%writefile api/main.py
from fastapi import FastAPI
import pandas as pd
import os

app = FastAPI(title="API Data Pipeline Bencinera - CNE")

@app.get("/")
def index():
    return {"estado": "Operativo", "arquitectura": "ETL + FastAPI Dockerizados", "rol": "Backend"}

@app.get("/api/consumo-industria")
def consumo_por_industria():
    path = "data/transacciones_limpias.csv"
    if not os.path.exists(path):
        return {"error": "Ejecuta el ETL primero para generar la data histórica"}

    df = pd.read_csv(path)
    # Agrupamos por los sectores industriales
    resumen = df.groupby('industria')['cantidad'].sum().reset_index()
    return resumen.to_dict(orient="records")

@app.get("/api/historico-combustibles")
def historico_combustibles():
    path = "data/transacciones_limpias.csv"
    if not os.path.exists(path):
        return {"error": "Dataset no disponible"}

    df = pd.read_csv(path)
    # Filtro avanzado para que Vicente grafique el consumo por tipo de producto
    resumen = df.groupby('nombre_prod')['cantidad'].sum().reset_index()
    return resumen.to_dict(orient="records")

@app.get("/api/precios-nacionales")
def precios_promedio():
    path = "data/precios_cne.csv"
    if not os.path.exists(path):
        return {"error": "Telemetría CNE no sincronizada"}

    df = pd.read_csv(path)
    promedios = {}
    for col in ['precio_93', 'precio_95', 'diesel']:
        if col in df.columns:
            # Calculamos el promedio y lo redondeamos a 2 decimales
            promedios[col] = round(float(df[col].mean()), 2)

    return {"unidad": "CLP", "promedios": promedios}

Overwriting api/main.py


### Definición de la API (`api/main.py`)

Este archivo contiene la implementación de la API REST utilizando FastAPI. Define varios endpoints para servir datos procesados por el ETL, como el consumo por industria, el histórico de combustibles y los precios promedio nacionales.

## api/main.py - API FastAPI para Visualización de Datos

In [14]:
%%writefile tests/test_etl.py
import unittest
import pandas as pd
import os

class TestBencineraPipeline(unittest.TestCase):
    def test_existencia_data_limpia(self):
        """Valida que la fase de carga del ETL guardó físicamente el archivo procesado."""
        self.assertTrue(os.path.exists("data/transacciones_limpias.csv"))

    def test_calidad_esquema(self):
        """Verifica que el dataset procesado no contenga cantidades nulas en las transacciones."""
        # Primero revisamos si el archivo existe para evitar que este test falle por herencia
        if os.path.exists("data/transacciones_limpias.csv"):
            df = pd.read_csv("data/transacciones_limpias.csv")
            self.assertEqual(df['cantidad'].isnull().sum(), 0)
        else:
            self.skipTest("El archivo data/transacciones_limpias.csv no existe todavía.")

if __name__ == '__main__':
    unittest.main()

Writing tests/test_etl.py


### Pruebas Unitarias del ETL (`tests/test_etl.py`)

Este script define pruebas unitarias para el pipeline ETL, verificando la existencia del archivo de datos limpios y la calidad del esquema (por ejemplo, que no haya valores nulos en columnas críticas).

In [15]:
!mkdir -p tests
!touch tests/__init__.py

### Inicialización del Entorno de Pruebas

Este comando asegura que el directorio `tests` existe y contiene un archivo `__init__.py`, lo cual es necesario para que Python reconozca el directorio como un paquete y se puedan ejecutar las pruebas unitarias.

In [16]:
!python -m unittest tests/test_etl.py

..
----------------------------------------------------------------------
Ran 2 tests in 0.674s

OK


### Ejecución de Pruebas Unitarias

Este comando ejecuta las pruebas unitarias definidas en `tests/test_etl.py`, proporcionando una verificación automatizada de la funcionalidad del pipeline ETL.

In [18]:
%%writefile dashboards/app.py
import streamlit as st
import pandas as pd
import requests

st.set_page_config(page_title="Plataforma de Combustibles", layout="wide")

# Conexión interna a través de la red de Docker (apunta al contenedor de tu API)
API_URL = "http://backend_api:8000"

st.title("⛽ Análisis de Dispensación y Precios")

# Selector Multiaudiencia exigido por la rúbrica
st.sidebar.header("Perfil de Usuario")
audiencia = st.sidebar.radio("Vista:", ["Ejecutiva (Estratégica)", "Operativa (Técnica)"])

def fetch_data(endpoint):
    try:
        return requests.get(f"{API_URL}{endpoint}").json()
    except:
        return None

if audiencia == "Ejecutiva (Estratégica)":
    st.markdown("### 📈 Resumen Financiero y de Mercado")
    st.info("Métricas clave para la toma de decisiones gerenciales.")

    precios = fetch_data("/api/precios-nacionales")
    if precios and "promedios" in precios:
        c1, c2, c3 = st.columns(3)
        c1.metric("Promedio Nacional 93", f"${precios['promedios'].get('precio_93', 0)}")
        c2.metric("Promedio Nacional 95", f"${precios['promedios'].get('precio_95', 0)}")
        c3.metric("Promedio Diésel", f"${precios['promedios'].get('diesel', 0)}")

    st.divider()
    st.markdown("#### Volumen de Ventas por Industria")
    data_ind = fetch_data("/api/consumo-industria")
    if data_ind:
        df_ind = pd.DataFrame(data_ind)
        if not df_ind.empty:
            st.bar_chart(df_ind.set_index("industria")["cantidad"], color="#1f77b4")

else:
    st.markdown("### ⚙️ Telemetría y Desglose Operativo")
    st.info("Registro tabular y distribución volumétrica para mantención de equipos.")

    data_hist = fetch_data("/api/historico-combustibles")
    if data_hist:
        df_hist = pd.DataFrame(data_hist)
        if not df_hist.empty:
            col1, col2 = st.columns([1, 2])
            with col1:
                st.markdown("#### Registro Tabular")
                st.dataframe(df_hist, use_container_width=True)
            with col2:
                st.markdown("#### Dispensación por Producto")
                st.bar_chart(df_hist.set_index("nombre_prod")["cantidad"], color="#ff7f0e")

Writing dashboards/app.py


In [19]:
%%writefile docker-compose.yml
version: '3.8'

services:
  backend_api:
    build: .
    container_name: bencinera_api
    ports:
      - "8000:8000"
    volumes:
      - ./data:/app/data
    environment:
      # Inyección segura de credenciales
      - CNE_USER=je.urquiaga@duocuc.cl
      - CNE_PASSWORD=Joder123#

  frontend_dashboard:
    image: python:3.11-slim
    container_name: bencinera_dashboard
    working_dir: /app
    volumes:
      - ./dashboards:/app/dashboards
    command: >
      sh -c "pip install streamlit pandas requests &&
      streamlit run dashboards/app.py --server.port 8501 --server.address 0.0.0.0"
    ports:
      - "8501:8501"
    depends_on:
      - backend_api

Writing docker-compose.yml
